In [1]:
# Import or install Sionna
try:
    import sionna.rt
except ImportError as e:
    import os
    os.system("pip install sionna-rt")
    import sionna.rt

# Other imports
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import drjit as dr
import mitsuba as mi

no_preview = False # Toggle to False to use the preview widget


%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
    
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, ITURadioMaterial,\
    Camera, PathSolver, InteractionType, RadioMapSolver
from sionna.rt.utils import r_hat

In [2]:
import os
import mitsuba as mi
from sionna.rt import load_scene

# ==============================================================================
# 1. XML 파일 경로 보정 및 로드
# ==============================================================================
# 원본 파일 경로 (사용자 환경에 맞게 설정)
original_xml_path = "/data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_itu.xml"
scene_dir = os.path.dirname(original_xml_path)
fixed_xml_path = os.path.join(scene_dir, "kookmin_fixed_temp.xml") # 임시 수정 파일

# 1) 원본 XML 읽기
with open(original_xml_path, 'r', encoding='utf-8') as f:
    xml_content = f.read()

# 2) 상대 경로("meshes/")를 절대 경로로 치환하여 "파일 찾기 실패" 방지
#    (예: "meshes/file.ply" -> "/data1/mh/.../meshes/file.ply")
abs_mesh_path = os.path.join(scene_dir, "meshes") + "/"
xml_content_fixed = xml_content.replace('value="meshes/', f'value="{abs_mesh_path}')

# 3) 수정된 내용을 임시 파일로 저장
with open(fixed_xml_path, 'w', encoding='utf-8') as f:
    f.write(xml_content_fixed)

print(f"[준비] 경로가 수정된 임시 XML 파일을 생성했습니다: {fixed_xml_path}")

# 4) 장면 로드
try:
    scene = load_scene(fixed_xml_path)
    print("[성공] 장면(Scene)을 성공적으로 불러왔습니다.")
except Exception as e:
    print(f"[오류] 장면 로드 실패: {e}")

# ==============================================================================
# 2. 재질(Material) 정보 확인
# ==============================================================================
if 'scene' in globals():
    print("\n" + "="*60)
    print(f"{'Object Name':<30} | {'Assigned Material':<20}")
    print("="*60)
    
    # 씬에 있는 모든 객체를 순회하며 할당된 재질 확인
    for name, obj in scene.objects.items():
        # 재질 객체가 있으면 이름 출력, 없으면 None
        mat_name = obj.radio_material.name if obj.radio_material else "None"
        print(f"{name:<30} | {mat_name:<20}")
        
    print("="*60)
    
    # 정의된 모든 재질(Radio Material) 목록 확인
    print("\n[정의된 재질 목록]")
    for mat_name, mat in scene.radio_materials.items():
        print(f" - 이름: {mat_name:<15} (Type: {mat.itu_type}, Thickness: {mat.thickness})")

[준비] 경로가 수정된 임시 XML 파일을 생성했습니다: /data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_fixed_temp.xml
[성공] 장면(Scene)을 성공적으로 불러왔습니다.

Object Name                    | Assigned Material   
elm__23                        | wall                
elm__24                        | roof                
elm__25                        | 8b4513              
elm__26                        | 2f4f4f              
elm__27                        | red                 
elm__28                        | white               
elm__29                        | gray                
elm__30                        | black               
elm__31                        | darkgrey            
elm__32                        | grey                
elm__33                        | lightgrey           
elm__34                        | silver              
elm__35                        | brown               
elm__36                        | d2aa6d              
elm__37                        | a58e9a              
elm_

In [3]:
# ==============================================================================
# 1. XML 경로 수정 및 안전한 로드
# ==============================================================================
# 원본 파일 및 폴더 경로 (사용자 환경)
xml_path = "/data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_itu.xml"
scene_dir = os.path.dirname(xml_path)
meshes_dir = os.path.join(scene_dir, "meshes")
temp_xml_path = os.path.join(scene_dir, "kookmin_fixed_absolute.xml")

# 1) XML 파일 읽기
with open(xml_path, 'r', encoding='utf-8') as f:
    xml_content = f.read()

# 2) 상대 경로("meshes/")를 절대 경로("/data1/.../meshes/")로 치환
if os.path.exists(meshes_dir):
    abs_mesh_path = meshes_dir + "/" if not meshes_dir.endswith("/") else meshes_dir
    xml_content_fixed = xml_content.replace('value="meshes/', f'value="{abs_mesh_path}')
    
    # 3) 임시 파일로 저장
    with open(temp_xml_path, 'w', encoding='utf-8') as f:
        f.write(xml_content_fixed)
    print(f"[설정] 경로가 수정된 임시 XML 생성: {temp_xml_path}")
else:
    raise FileNotFoundError(f"meshes 폴더를 찾을 수 없습니다: {meshes_dir}")

# 4) load_scene으로 로드
try:
    scene = load_scene(temp_xml_path)
    print("[성공] 장면(Scene) 로드 완료.")
except Exception as e:
    print(f"[치명적 오류] 장면 로드 실패: {e}")
    raise e

#

[설정] 경로가 수정된 임시 XML 생성: /data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_fixed_absolute.xml
[성공] 장면(Scene) 로드 완료.


In [4]:
# ==============================================================================
# 2. 도로 재질 변경 (시각화용)
# ==============================================================================
red_road_mat = ITURadioMaterial(name="red_road_mat",
                                itu_type="concrete",
                                thickness=0.2,
                                color=[1.0, 0.0, 0.0])
scene.add(red_road_mat)

target_road_id = "elm__00"
if target_road_id in scene.objects:
    scene.objects[target_road_id].radio_material = red_road_mat
    print(f"[설정] 도로({target_road_id})를 빨간색으로 변경했습니다.")

[설정] 도로(elm__00)를 빨간색으로 변경했습니다.


In [5]:
road_object_id = "elm__00"
road_positions = []

print(f"[탐색] Mitsuba Scene 내부에서 '{road_object_id}' 형상을 찾습니다...")

if hasattr(scene, 'mi_scene'):
    mi_scene = scene.mi_scene
    
    # 1. Mitsuba Scene의 모든 Shape를 순회하며 ID 매칭
    target_shape = None
    for s in mi_scene.shapes():
        if s.id() == road_object_id:
            target_shape = s
            break
    
    if target_shape: 
        try:           
            params = mi.traverse(target_shape)
                        
            if 'vertex_positions' in params:
                vertex_buffer = params['vertex_positions']
                
                # NumPy 변환 (1차원 배열: x, y, z, x, y, z ...)
                vertices_flat = np.array(vertex_buffer)
                
                # (N, 3) 형태로 변환 (x, y, z)
                if len(vertices_flat) > 0:
                    vertices = vertices_flat.reshape(-1, 3)
                    road_positions = vertices
                    print(f"  -> 추출된 도로 좌표 수: {len(road_positions)}개")
                else:
                    print("  [경고] 버퍼가 비어있습니다.")
            else:
                print(f"[오류] '{road_object_id}' 객체에 'vertex_positions' 속성이 없습니다.")

        except Exception as e:
            print(f"[오류] Vertex 추출 중 에러 발생: {e}")
    else:
        print(f"[실패] ID가 '{road_object_id}'인 Shape를 mi_scene에서 찾을 수 없습니다.")
else:
    print("[오류] scene 객체에서 'mi_scene' 속성을 찾을 수 없습니다.")

[탐색] Mitsuba Scene 내부에서 'elm__00' 형상을 찾습니다...
  -> 추출된 도로 좌표 수: 610개


In [ ]:
# ==============================================================================
# 3. 경로(Trajectory) 좌표 정밀 보정 (Raw Data Inspection)
# ==============================================================================
print("[보정] 빨간색 도로 좌표 정밀 분석 시작...")

road_positions = []
target_road_id = "elm__00"

if hasattr(scene, 'mi_scene'):
    for s in scene.mi_scene.shapes():
        if target_road_id in s.id():
            params = mi.traverse(s)
            if 'vertex_positions' in params:
                # 1) 원본 좌표 추출
                v_pos = np.array(params['vertex_positions'], dtype=np.float32).reshape(-1, 3)
                
                # 2) [진단] 412번 인덱스의 '원본(Raw)' 좌표 확인
                if len(v_pos) > 412:
                    raw_412 = v_pos[412]
                    print(f" -> [진단] Raw Index 412: {raw_412}")
                    # 예상: [418.xx, 0.0, -308.xx] 또는 [418.xx, -308.xx, 0.0] 등
                    
                    # 3) [해결] 목표 좌표(Target)와 비교하여 매핑 결정
                    # Target Z: -308.20
                    
                    # Case A: Raw Y가 -308 근처인 경우 -> Y를 Z로 (x, 0, y)
                    if np.isclose(raw_412[1], -308.2, atol=5.0):
                        print(" -> [결정] Y축 데이터를 Z축으로 이동합니다. (Y -> Z)")
                        road_positions = np.zeros_like(v_pos)
                        road_positions[:, 0] = v_pos[:, 0]
                        road_positions[:, 1] = 0.0
                        road_positions[:, 2] = v_pos[:, 1]
                        
                    # Case B: Raw Z가 -308 근처인 경우 -> 그대로 사용 (Z -> Z)
                    elif np.isclose(raw_412[2], -308.2, atol=5.0):
                        print(" -> [결정] Z축 데이터를 그대로 사용합니다. (No Rotation)")
                        road_positions = np.zeros_like(v_pos)
                        road_positions[:, 0] = v_pos[:, 0]
                        road_positions[:, 1] = 0.0
                        road_positions[:, 2] = v_pos[:, 2]
                        
                    # Case C: Raw Y가 308 근처인 경우 -> 부호 반전 후 이동 (-Y -> Z)
                    elif np.isclose(raw_412[1], 308.2, atol=5.0):
                        print(" -> [결정] Y축 데이터를 반전하여 Z축으로 이동합니다. (-Y -> Z)")
                        road_positions = np.zeros_like(v_pos)
                        road_positions[:, 0] = v_pos[:, 0]
                        road_positions[:, 1] = 0.0
                        road_positions[:, 2] = -v_pos[:, 1]

                    # Case D: Raw Z가 308 근처인 경우 -> 부호 반전 (-Z -> Z)
                    elif np.isclose(raw_412[2], 308.2, atol=5.0):
                        print(" -> [결정] Z축 데이터를 반전하여 사용합니다. (-Z -> Z)")
                        road_positions = np.zeros_like(v_pos)
                        road_positions[:, 0] = v_pos[:, 0]
                        road_positions[:, 1] = 0.0
                        road_positions[:, 2] = -v_pos[:, 2]
                        
                    else:
                        print(" -> [경고] 자동 매핑 실패. Raw 데이터가 예상과 다릅니다. 원본 그대로 사용합니다.")
                        # 기본: (x, 0, y) 시도 (가장 흔한 패턴)
                        road_positions = np.zeros_like(v_pos)
                        road_positions[:, 0] = v_pos[:, 0]
                        road_positions[:, 1] = 0.0
                        road_positions[:, 2] = v_pos[:, 1]

                print(f" -> 좌표 변환 완료 ({len(road_positions)} vertices)")
            break

bs 1
ue 7
V 

In [7]:
import os
import tensorflow as tf
import numpy as np
import sionna

# [버전 호환성] Import 경로 처리
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, PathSolver
try:
    from sionna.phy.ofdm import ResourceGrid
except ImportError:
    from sionna.ofdm import ResourceGrid

# ==========================================
# 1. 파일 저장 경로 설정 (요청 사항 반영)
# ==========================================
# 저장할 디렉토리 경로
output_dir = "/data1/mh/sionna/tutorials/rt/build code/paper/L1_data"

# 디렉토리가 없으면 생성
if not os.path.exists(output_dir):
    try:
        os.makedirs(output_dir)
        print(f"디렉토리 생성됨: {output_dir}")
    except OSError as e:
        print(f"[오류] 디렉토리를 생성할 수 없습니다: {e}")
        # 실패 시 현재 디렉토리에 저장하도록 fallback
        output_dir = "."

# Scene 로딩을 위한 경로 설정 (기존 유지)
xml_path = "/data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_itu.xml"
scene_dir = os.path.dirname(xml_path)
temp_xml_path = os.path.join(scene_dir, "kookmin_fixed_absolute.xml")

# ==========================================
# 2. 시스템 및 시뮬레이션 파라미터 (정밀 설정)
# ==========================================
carrier_frequency = 3.5e9
subcarrier_spacing = 30e3 
fft_size = 72
num_ofdm_symbols = 14

# [수정됨] 정밀 시뮬레이션 설정
simulation_time = 10.0 # 10초
dt = 0.5e-3 # 0.5 ms (30kHz SCS의 1 Slot 길이)
total_steps = int(simulation_time / dt)

print(f"--- 시뮬레이션 설정 ---")
print(f"총 시간: {simulation_time}초")
print(f"시간 간격(dt): {dt}초 (0.5ms)")
print(f"총 스텝 수: {total_steps} (메모리 주의)")

# ==========================================
# 3. 경로 및 기지국 설정
# ==========================================
# 기지국(BS) 위치
bs_position = [323.47, 36.87, -204.31]

# 이동성 설정
num_ues = 8
speeds = [10, 15, 20, 25, 30, 40, 50, 60]

# [필수] 이전 셀에서 생성된 road_positions 확인
if 'road_positions' not in globals() or len(road_positions) == 0:
    raise ValueError("메모리에 'road_positions' 변수가 없습니다. 도로 좌표 추출 코드를 먼저 실행해주세요.")

path_indices = [
    412, 410, 408, 406, 404, 401, 400, 72, 69, 68, 342, 340, 338, 335, 334, 
    88, 86, 84, 81, 464, 462, 459, 295, 370, 294, 368, 366, 363, 308, 306, 
    303, 302, 514, 511, 510, 488, 483, 482, 172, 269, 292, 288, 286, 284, 
    281, 604, 601, 507, 289, 506, 268, 522, 517, 516, 122, 120, 117, 116, 
    525, 582
]
trajectory_points = road_positions[path_indices]

# PolylineWalker 클래스 정의
class PolylineWalker:
    def __init__(self, points):
        self.points = points
        diffs = points[1:] - points[:-1]
        self.seg_lengths = np.linalg.norm(diffs, axis=1)
        self.cum_dist = np.insert(np.cumsum(self.seg_lengths), 0, 0.0)
        self.total_length = self.cum_dist[-1]
        
    def get_position(self, distance):
        if distance >= self.total_length: return self.points[-1]
        if distance <= 0: return self.points[0]
        idx = np.searchsorted(self.cum_dist, distance) - 1
        idx = max(0, idx)
        p_start = self.points[idx]
        p_end = self.points[idx+1]
        seg_len = self.seg_lengths[idx]
        ratio = (distance - self.cum_dist[idx]) / seg_len if seg_len > 0 else 0
        return p_start + (p_end - p_start) * ratio

walker = PolylineWalker(trajectory_points)

# ==========================================
# 4. Scene 구성
# ==========================================
scene = load_scene(temp_xml_path)

# 안테나: BS(8x8), UE(1x1)
bs_array = PlanarArray(num_rows=8, num_cols=8, vertical_spacing=0.5, horizontal_spacing=0.5, pattern="iso", polarization="V")
ue_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="V")

scene.tx_array = bs_array
scene.rx_array = ue_array

# 기기 배치
tx = Transmitter(name="BS", position=bs_position, orientation=[0,0,0])
scene.add(tx)

ues = []
start_pos = trajectory_points[0]
for i in range(num_ues):
    rx = Receiver(name=f"UE_{i}", position=start_pos, orientation=[0,0,0])
    scene.add(rx)
    ues.append(rx)

# ==========================================
# 5. 시뮬레이션 루프 (대용량 데이터 생성)
# ==========================================
solver = PathSolver()
dataset_h = [] # 여기에 20,000개의 데이터가 쌓입니다.

print("시뮬레이션 시작...")

for step in range(total_steps):
    current_time = step * dt
    
    # 이동
    for i, ue in enumerate(ues):
        speed = speeds[i]
        new_pos = walker.get_position(speed * current_time)
        ue.position = new_pos
    
    # Ray Tracing
    paths = solver(scene, max_depth=3)
    
    # CFR 계산
    frequencies = subcarrier_spacing * tf.range(fft_size, dtype=tf.float32)
    cfr_output = paths.cfr(frequencies=frequencies)
    
    if isinstance(cfr_output, tuple):
        h_freq = cfr_output[0]
    else:
        h_freq = cfr_output
        
    dataset_h.append(h_freq.numpy())
    
    # 진행 상황 출력 (너무 자주 출력하지 않도록 1000스텝마다)
    if step % 1000 == 0:
        print(f"Progress: {step}/{total_steps} ({(step/total_steps)*100:.1f}%)")

# ==========================================
# 6. 저장
# ==========================================
print("데이터 변환 중... (시간이 소요될 수 있습니다)")
dataset_h = np.array(dataset_h)
dataset_h = np.squeeze(dataset_h)

print(f"최종 데이터 형태: {dataset_h.shape}")
# 예상: (20000, 8, 64, 72)

file_name = "vit_channel_dataset_precise_10s.npy"
save_path = os.path.join(output_dir, file_name)

np.save(save_path, dataset_h)
print(f"저장 완료: {save_path}")

2026-02-02 13:49:03.095543: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770007743.108281   30894 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770007743.112173   30894 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770007743.122910   30894 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770007743.122921   30894 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770007743.122923   30894 computation_placer.cc:177] computation placer alr

--- 시뮬레이션 설정 ---
총 시간: 10.0초
시간 간격(dt): 0.0005초 (0.5ms)
총 스텝 수: 20000 (메모리 주의)
시뮬레이션 시작...


I0000 00:00:1770007747.374543   30894 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 164 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:81:00.0, compute capability: 8.6
I0000 00:00:1770007747.375733   30894 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22057 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:c1:00.0, compute capability: 8.6


Progress: 0/20000 (0.0%)


KeyboardInterrupt: 

In [6]:
import os
import tensorflow as tf
import numpy as np
import drjit as dr
import gc
import sionna
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, PathSolver

# ==========================================
# 1. 기본 설정 (60초, 단일 주파수)
# ==========================================
# 저장 경로
output_dir = "/data1/mh/sionna/tutorials/rt/build code/paper/L1_data"
if not os.path.exists(output_dir):
    try: os.makedirs(output_dir)
    except: output_dir = "."

# 맵 파일 경로
xml_path = "/data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_itu.xml"
scene_dir = os.path.dirname(xml_path)
temp_xml_path = os.path.join(scene_dir, "kookmin_fixed_absolute.xml")

# 시뮬레이션 시간 및 주파수 설정
simulation_time = 60.0
dt = 0.5e-3
total_steps = int(simulation_time / dt)
frequencies = tf.cast([0], tf.float32) # 단일 주파수 (메모리 절약)

# UE 속도 및 개수 설정
speeds_kmh = [20, 40, 60, 80, 100, 120]
speeds_ms = [v / 3.6 for v in speeds_kmh]
num_ues = len(speeds_kmh)

# 도로 좌표 확인
if 'road_positions' not in globals() or len(road_positions) == 0:
    raise ValueError("도로 좌표(road_positions)가 메모리에 없습니다.")

path_indices = [
    412, 410, 408, 406, 404, 401, 400, 72, 69, 68, 342, 340, 338, 335, 334, 
    88, 86, 84, 81, 464, 462, 459, 295, 370, 294, 368, 366, 363, 308, 306, 
    303, 302, 514, 511, 510, 488, 483, 482, 172, 269, 292, 288, 286, 284, 
    281, 604, 601, 507, 289, 506, 268, 522, 517, 516, 122, 120, 117, 116, 
    525, 582
]
trajectory_points = road_positions[path_indices]

# 이동 경로 계산기
class PolylineWalker:
    def __init__(self, points):
        self.points = points
        diffs = points[1:] - points[:-1]
        self.seg_lengths = np.linalg.norm(diffs, axis=1)
        self.cum_dist = np.insert(np.cumsum(self.seg_lengths), 0, 0.0)
        self.total_length = self.cum_dist[-1]
    def get_position(self, distance):
        if distance >= self.total_length: return self.points[-1]
        if distance <= 0: return self.points[0]
        idx = np.searchsorted(self.cum_dist, distance) - 1
        idx = max(0, idx)
        p_start = self.points[idx]
        p_end = self.points[idx+1]
        ratio = (distance - self.cum_dist[idx]) / self.seg_lengths[idx] if self.seg_lengths[idx] > 0 else 0
        return p_start + (p_end - p_start) * ratio

walker = PolylineWalker(trajectory_points)

# ==========================================
# 2. Scene 초기화 및 기지국 정보 설정
# ==========================================
scene = load_scene(temp_xml_path)

bs_array = PlanarArray(num_rows=8, num_cols=8, vertical_spacing=0.5, horizontal_spacing=0.5, pattern="iso", polarization="V")
ue_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="V")
scene.tx_array = bs_array
scene.rx_array = ue_array

# [수정됨] 3개의 기지국 좌표
tx_positions = [[-125.663, 56.367, -181.453], [323.472, 36.869, -204.315], [0.663, 56.367, -181.453]]
tx_names = ["Tx_1", "Tx_2", "Tx_3"]

# 계산용 더미 수신기 (Active_Rx) 배치
dummy_rx_name = "Active_Rx"
scene.add(Receiver(name=dummy_rx_name, position=trajectory_points[0], orientation=[0,0,0]))
active_rx = scene.receivers[dummy_rx_name]

# ==========================================
# 3. 시뮬레이션 (순차 처리로 메모리 보호)
# ==========================================
solver = PathSolver()
final_data_container = [] # 최종 데이터 담을 곳

print(f"🚀 [3-BS] 안정성 최적화 시뮬레이션 시작")
print(f"   - 총 UE: {num_ues}명, 총 BS: {len(tx_names)}개")
print(f"   - 시간: 60초 (Total Steps: {total_steps})")

# 1. UE 루프
for ue_idx in range(num_ues):
    ue_speed = speeds_ms[ue_idx]
    print(f"\n[UE {ue_idx+1}/{num_ues}] (속도 {speeds_kmh[ue_idx]} km/h) 처리 중...")
    
    ue_bs_data_list = [] # 현재 UE의 [BS1, BS2, BS3] 데이터를 모을 리스트
    
    # 2. 기지국(Tx) 루프 (하나씩 켜고 끄면서 계산)
    for tx_idx, (tx_name, tx_pos) in enumerate(zip(tx_names, tx_positions)):
        print(f"  ├─ 기지국 {tx_idx+1} ({tx_name}) 계산 중...", end="")
        
        # 현재 기지국 배치
        active_tx_name = "Active_Tx"
        if active_tx_name in scene.transmitters:
            scene.remove(active_tx_name)
        scene.add(Transmitter(name=active_tx_name, position=tx_pos, orientation=[0,0,0]))
        
        current_tx_time_data = []
        
        # 3. 시간(Time) 루프
        for step in range(total_steps):
            current_time = step * dt
            
            # 위치 이동
            new_pos = walker.get_position(ue_speed * current_time)
            active_rx.position = new_pos
            
            # Ray Tracing (샘플 수 조절로 안정성 확보)
            paths = solver(scene, max_depth=3, samples_per_src=40000)
            
            # CFR 계산
            cfr = paths.cfr(frequencies=frequencies)
            if isinstance(cfr, tuple): cfr = cfr[0]
            
            # 결과 저장 (Squeeze로 차원 축소: (64,))
            val = np.squeeze(cfr.numpy())
            current_tx_time_data.append(val)
            
        # 60초 데이터 완료 (Time, 64)
        tx_np_data = np.array(current_tx_time_data)
        ue_bs_data_list.append(tx_np_data)
        
        # 메모리 정리 (필수)
        scene.remove(active_tx_name)
        dr.flush_malloc_cache()
        gc.collect()
        print(" 완료.")

    # 한 UE의 모든 BS 데이터 통합 -> (BS, Time, 64) -> Transpose -> (Time, BS, 64)
    ue_combined = np.stack(ue_bs_data_list, axis=0)
    ue_combined = np.transpose(ue_combined, (1, 0, 2))
    
    final_data_container.append(ue_combined)
    print(f"  └─ UE {ue_idx+1} 완료.")

# ==========================================
# 4. 데이터 병합 및 저장
# ==========================================
print("\n전체 데이터 병합 중...")
# (UE, Time, BS, 64)
final_dataset = np.stack(final_data_container, axis=0)

# (Time, UE, BS, 64) 로 변경
final_dataset = np.transpose(final_dataset, (1, 0, 2, 3))

print(f"최종 데이터 형태: {final_dataset.shape}")
# 예상: (120000, 6, 3, 64)

file_name = "vit_channel_dataset_3BS_6UEs_60s_Final.npy"
save_path = os.path.join(output_dir, file_name)
np.save(save_path, final_dataset)

print(f"✅ 저장 완료: {save_path}")

2026-02-02 23:31:29.663856: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770042689.678939  229545 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770042689.683535  229545 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770042689.696886  229545 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770042689.696897  229545 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770042689.696898  229545 computation_placer.cc:177] computation placer alr

🚀 [3-BS] 안정성 최적화 시뮬레이션 시작
   - 총 UE: 6명, 총 BS: 3개
   - 시간: 60초 (Total Steps: 120000)

[UE 1/6] (속도 20 km/h) 처리 중...
  ├─ 기지국 1 (Tx_1) 계산 중...

KeyboardInterrupt: 

In [8]:
import os
import tensorflow as tf
import numpy as np
import drjit as dr
import gc
import sionna

# [버전 호환성] Import 경로 처리
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, PathSolver

# ==========================================
# 1. 파일 저장 경로 및 기본 설정
# ==========================================
output_dir = "/data1/mh/sionna/tutorials/rt/build code/paper/L1_data"
if not os.path.exists(output_dir):
    try: os.makedirs(output_dir)
    except: output_dir = "."

# Scene 경로
xml_path = "/data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_itu.xml"
scene_dir = os.path.dirname(xml_path)
temp_xml_path = os.path.join(scene_dir, "kookmin_fixed_absolute.xml")

# ==========================================
# 2. 시스템 파라미터 (60초, 3 BS)
# ==========================================
simulation_time = 60.0 # 60초
dt = 0.5e-3 # 0.5 ms
total_steps = int(simulation_time / dt)
frequencies = tf.cast([0], tf.float32) # 단일 주파수 (Central Freq)

# 기지국 3개 설정
tx_positions = [[-125.663, 56.367, -181.453], [323.472, 36.869, -204.315], [0.663, 56.367, -181.453]]
tx_names = ["Tx_1", "Tx_2", "Tx_3"]

# UE 설정 (속도별 6개)
speeds_kmh = [20, 40, 60, 80, 100, 120]
speeds_ms = [v / 3.6 for v in speeds_kmh]
num_ues = len(speeds_kmh)

print(f"--- 시뮬레이션 설정 ---")
print(f"총 시간: {simulation_time}초 ({total_steps} Steps)")
print(f"기지국 수: {len(tx_names)}개")
print(f"UE 수: {num_ues}명")

# ==========================================
# 3. 경로 및 Walker 설정
# ==========================================
# [중요] road_positions가 메모리에 있어야 합니다.
if 'road_positions' not in globals() or len(road_positions) == 0:
    # 만약 변수가 없다면 파일에서 로드 시도 (없으면 에러)
    if os.path.exists("road_positions.npy"):
        road_positions = np.load("road_positions.npy")
    else:
        raise ValueError("메모리에 'road_positions' 변수가 없습니다.")

path_indices = [
    412, 410, 408, 406, 404, 401, 400, 72, 69, 68, 342, 340, 338, 335, 334, 
    88, 86, 84, 81, 464, 462, 459, 295, 370, 294, 368, 366, 363, 308, 306, 
    303, 302, 514, 511, 510, 488, 483, 482, 172, 269, 292, 288, 286, 284, 
    281, 604, 601, 507, 289, 506, 268, 522, 517, 516, 122, 120, 117, 116, 
    525, 582
]
trajectory_points = road_positions[path_indices]

class PolylineWalker:
    def __init__(self, points):
        self.points = points
        diffs = points[1:] - points[:-1]
        self.seg_lengths = np.linalg.norm(diffs, axis=1)
        self.cum_dist = np.insert(np.cumsum(self.seg_lengths), 0, 0.0)
        self.total_length = self.cum_dist[-1]
    def get_position(self, distance):
        if distance >= self.total_length: return self.points[-1]
        if distance <= 0: return self.points[0]
        idx = np.searchsorted(self.cum_dist, distance) - 1
        idx = max(0, idx)
        p_start = self.points[idx]
        p_end = self.points[idx+1]
        ratio = (distance - self.cum_dist[idx]) / self.seg_lengths[idx] if self.seg_lengths[idx] > 0 else 0
        return p_start + (p_end - p_start) * ratio

walker = PolylineWalker(trajectory_points)

# ==========================================
# 4. Scene 구성
# ==========================================
scene = load_scene(temp_xml_path)

# 안테나 설정
bs_array = PlanarArray(num_rows=8, num_cols=8, vertical_spacing=0.5, horizontal_spacing=0.5, pattern="iso", polarization="V")
ue_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="V")
scene.tx_array = bs_array
scene.rx_array = ue_array

# 계산용 더미 Receiver 1개 배치 (위치만 계속 바꿈)
dummy_rx_name = "Active_Rx"
scene.add(Receiver(name=dummy_rx_name, position=trajectory_points[0], orientation=[0,0,0]))
active_rx = scene.receivers[dummy_rx_name]

solver = PathSolver()

# ==========================================
# 5. 시뮬레이션 루프 (UE -> BS -> Time 순서)
# ==========================================
# 데이터를 한 번에 다 모으면 RAM이 터질 수 있으므로, UE 하나 끝날 때마다 부분 저장합니다.
print("🚀 시뮬레이션 시작...")

final_ue_data_list = [] # 나중에 합치기 위한 리스트

for ue_idx in range(num_ues):
    ue_speed = speeds_ms[ue_idx]
    print(f"\n▶ [UE {ue_idx+1}/{num_ues}] (속도 {speeds_kmh[ue_idx]} km/h) 처리 시작...")
    
    # 이 UE에 대한 3개 기지국 데이터를 모을 리스트
    current_ue_bs_data = [] 

    # 기지국 3개를 하나씩 켜고 끄면서 계산 (메모리 절약)
    for tx_idx, (tx_name, tx_pos) in enumerate(zip(tx_names, tx_positions)):
        print(f"  └─ BS {tx_idx+1}/{len(tx_names)} 계산 중...", end="")
        
        # 현재 기지국만 맵에 추가
        active_tx_name = "Active_Tx"
        if active_tx_name in scene.transmitters:
            scene.remove(active_tx_name)
        scene.add(Transmitter(name=active_tx_name, position=tx_pos, orientation=[0,0,0]))
        
        # 60초 데이터 담을 리스트
        tx_time_data = []
        
        for step in range(total_steps):
            current_time = step * dt
            
            # 1. 위치 이동
            new_pos = walker.get_position(ue_speed * current_time)
            active_rx.position = new_pos
            
            # 2. Ray Tracing (샘플 수 2만개 제한 - 안정성)
            paths = solver(scene, max_depth=3, samples_per_src=20000)
            
            # 3. Channel Calculation
            cfr = paths.cfr(frequencies=frequencies)
            if isinstance(cfr, tuple): cfr = cfr[0]
            
            # 4. 데이터 추출 (Squeeze로 (1,1,1,64,1) -> (64,) 변환)
            val = np.squeeze(cfr.numpy())
            tx_time_data.append(val)
            
            # 5. 메모리 청소 (Dr.Jit 캐시 비우기 - 중요!)
            del paths, cfr
            if step % 2000 == 0:
                dr.flush_malloc_cache()
                gc.collect()
        
        # 60초 데이터 완료 -> Numpy 변환
        tx_np = np.array(tx_time_data) # Shape: (Time, 64)
        current_ue_bs_data.append(tx_np)
        
        # 기지국 제거 및 메모리 정리
        scene.remove(active_tx_name)
        dr.flush_malloc_cache()
        gc.collect()
        print(" 완료.")

    # ------------------------------------------
    # UE 1명 완료 시점: 데이터 통합 및 임시 저장
    # ------------------------------------------
    # (3, Time, 64) -> (Time, 3, 64)로 변환
    ue_combined = np.stack(current_ue_bs_data, axis=0) # [BS, Time, Ant]
    ue_combined = np.transpose(ue_combined, (1, 0, 2)) # [Time, BS, Ant]
    
    # 전체 리스트에 추가 (메모리가 충분하다면)
    final_ue_data_list.append(ue_combined)
    
    # [안전장치] 혹시 모르니 개별 파일로도 저장 (RAM 부족 시 복구용)
    part_filename = f"part_data_UE_{ue_idx}.npy"
    np.save(os.path.join(output_dir, part_filename), ue_combined)
    print(f"  💾 UE {ue_idx} 데이터 저장 완료 ({part_filename})")

# ==========================================
# 6. 최종 병합 및 저장
# ==========================================
print("\n📊 전체 데이터 병합 중...")
# final_ue_data_list 구조: [ (Time, 3, 64), (Time, 3, 64), ... ]
# Stack -> (UE, Time, 3, 64)
final_dataset = np.stack(final_ue_data_list, axis=0)

# 최종 차원 변경: (Time, UE, BS, Ant)
final_dataset = np.transpose(final_dataset, (1, 0, 2, 3))

print(f"최종 데이터 형태: {final_dataset.shape}")
# 예상: (120000, 6, 3, 64)

save_path = os.path.join(output_dir, "vit_channel_dataset_3BS_6UEs_60s_Sequential.npy")
np.save(save_path, final_dataset)
print(f"✅ 최종 파일 저장 완료: {save_path}")

--- 시뮬레이션 설정 ---
총 시간: 60.0초 (120000 Steps)
기지국 수: 3개
UE 수: 6명
🚀 시뮬레이션 시작...

▶ [UE 1/6] (속도 20 km/h) 처리 시작...
  └─ BS 1/3 계산 중...

jit_flush_malloc_cache(): Dr.Jit exhausted the available memory and had to flush its allocation cache to free up additional memory. This is an expensive operation and will have a negative effect on performance. You may want to change your computation so that it uses less memory. This warning will only be displayed once.


RuntimeError: jit_malloc(): out of memory! Could not allocate 4194304 bytes of device memory.

In [ ]:
import os
import tensorflow as tf
import numpy as np
import drjit as dr
import gc
import sionna

# [버전 호환성] Import
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, PathSolver

# ==========================================
# 1. 파일 저장 경로 및 기본 설정
# ==========================================
output_dir = "/data1/mh/sionna/tutorials/rt/build code/paper/L1_data"
if not os.path.exists(output_dir):
    try: os.makedirs(output_dir)
    except: output_dir = "."

# 저장할 파일명 (복소수 버전)
file_name = "vit_channel_dataset_precise_10s_complex_final.npy"
save_path = os.path.join(output_dir, file_name)

# Scene 경로
xml_path = "/data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_itu.xml"
scene_dir = os.path.dirname(xml_path)
temp_xml_path = os.path.join(scene_dir, "kookmin_fixed_absolute.xml")

# ==========================================
# 2. 시스템 파라미터
# ==========================================
simulation_time = 10.0 # 10초
dt = 0.5e-3 # 0.5 ms
total_steps = int(simulation_time / dt)

carrier_frequency = 3.5e9
subcarrier_spacing = 30e3 
fft_size = 72
frequencies = subcarrier_spacing * tf.range(fft_size, dtype=tf.float32)

# 기지국 3개 위치
tx_positions = [[-125.663, 56.367, -181.453], [323.472, 36.869, -204.315], [0.663, 56.367, -181.453]]
tx_names = ["Tx_1", "Tx_2", "Tx_3"]

# UE 설정 (속도별 8개)
speeds_kmh = [10, 15, 20, 25, 30, 40, 50, 60]
speeds_ms = [v / 3.6 for v in speeds_kmh]
num_ues = len(speeds_kmh)

print(f"--- 시뮬레이션 설정 ---")
print(f"총 시간: {simulation_time}초 ({total_steps} Steps)")
print(f"기지국 수: {len(tx_names)}개")
print(f"UE 수: {num_ues}명")
print(f"처리 방식: 1 Tx - 1 Rx 완전 순차 처리 (메모리 최적화)")

# ==========================================
# 3. 경로 및 Walker 설정
# ==========================================
if 'road_positions' not in globals():
    if os.path.exists("road_positions.npy"):
        road_positions = np.load("road_positions.npy")
    else:
        raise ValueError("메모리에 'road_positions' 변수가 없습니다.")

path_indices = [
    412, 410, 408, 406, 404, 401, 400, 72, 69, 68, 342, 340, 338, 335, 334, 
    88, 86, 84, 81, 464, 462, 459, 295, 370, 294, 368, 366, 363, 308, 306, 
    303, 302, 514, 511, 510, 488, 483, 482, 172, 269, 292, 288, 286, 284, 
    281, 604, 601, 507, 289, 506, 268, 522, 517, 516, 122, 120, 117, 116, 
    525, 582
]
trajectory_points = road_positions[path_indices]

class PolylineWalker:
    def __init__(self, points):
        self.points = points
        diffs = points[1:] - points[:-1]
        self.seg_lengths = np.linalg.norm(diffs, axis=1)
        self.cum_dist = np.insert(np.cumsum(self.seg_lengths), 0, 0.0)
        self.total_length = self.cum_dist[-1]
    def get_position(self, distance):
        if distance >= self.total_length: return self.points[-1]
        if distance <= 0: return self.points[0]
        idx = np.searchsorted(self.cum_dist, distance) - 1
        idx = max(0, idx)
        p_start = self.points[idx]
        p_end = self.points[idx+1]
        ratio = (distance - self.cum_dist[idx]) / self.seg_lengths[idx] if self.seg_lengths[idx] > 0 else 0
        return p_start + (p_end - p_start) * ratio

walker = PolylineWalker(trajectory_points)

# ==========================================
# 4. Scene 구성 (빈 껍데기만 로드)
# ==========================================
scene = load_scene(temp_xml_path)

bs_array = PlanarArray(num_rows=8, num_cols=8, vertical_spacing=0.5, horizontal_spacing=0.5, pattern="iso", polarization="V")
ue_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="V")
scene.tx_array = bs_array
scene.rx_array = ue_array

solver = PathSolver()

# ==========================================
# 5. 시뮬레이션 실행 (완전 순차 처리)
# ==========================================
# 최종 데이터를 담을 리스트
all_ue_data = []

print(f"🚀 시뮬레이션 시작...")

for ue_idx in range(num_ues):
    ue_speed = speeds_ms[ue_idx]
    print(f"\n▶ [UE {ue_idx+1}/{num_ues}] (속도 {speeds_kmh[ue_idx]} km/h) 처리 시작...")
    
    # 현재 UE의 [BS1, BS2, BS3] 데이터를 담을 리스트
    current_ue_bs_data = [] 
    
    # 기지국 순차 처리
    for tx_idx, (tx_name, tx_pos) in enumerate(zip(tx_names, tx_positions)):
        print(f"  └─ BS {tx_idx+1}/{len(tx_names)} 계산 중...", end="")
        
        # 1. 맵 청소
        if "Active_Tx" in scene.transmitters: scene.remove("Active_Tx")
        if "Active_Rx" in scene.receivers: scene.remove("Active_Rx")
        
        # 2. 현재 Tx와 Rx 1개씩만 추가
        scene.add(Transmitter(name="Active_Tx", position=tx_pos, orientation=[0,0,0]))
        scene.add(Receiver(name="Active_Rx", position=trajectory_points[0], orientation=[0,0,0]))
        
        active_rx = scene.receivers["Active_Rx"]
        
        # 60초(10초) 데이터 담을 리스트
        time_step_data = []
        
        for step in range(total_steps):
            current_time = step * dt
            
            # [핵심 수정] 위치 이동 (지면 + 1.5m 높이)
            ground_pos = walker.get_position(ue_speed * current_time)
            new_pos = ground_pos + np.array([0, 0, 1.5]) # 높이 보정
            active_rx.position = new_pos
            
            # [핵심 수정] Ray Tracing (샘플 수 100,000개로 증가)
            paths = solver(scene, max_depth=3, samples_per_src=100000)
            
            # CFR 계산
            cfr_output = paths.cfr(frequencies=frequencies)
            
            if isinstance(cfr_output, tuple): h_freq = cfr_output[0]
            else: h_freq = cfr_output
                
            # NumPy 변환
            h_freq_np = h_freq.numpy()
            
            # [진단: 첫 스텝만]
            if step == 0 and ue_idx == 0 and tx_idx == 0:
                print(f"\n   [진단] Shape: {h_freq_np.shape}, Type: {h_freq_np.dtype}")
                # 에너지 확인
                total_energy = np.sum(np.abs(h_freq_np))
                print(f"   [진단] 감지된 신호 총량: {total_energy:.6f}")
                
                if total_energy == 0:
                    print("   ⚠️ [치명적 경고] 여전히 경로를 못 찾았습니다. 맵 좌표 단위를 확인해야 할 수도 있습니다.")
                else:
                    print("   ✅ [성공] 경로가 감지되었습니다.")

            # [핵심 수정] 복소수 합치기 (만약 분리된 경우)
            if not np.iscomplexobj(h_freq_np) and h_freq_np.shape[-1] == 2:
                h_freq_np = h_freq_np[..., 0] + 1j * h_freq_np[..., 1]
            
            # (1,1,1,64) -> (64,) 로 줄여서 저장
            time_step_data.append(np.squeeze(h_freq_np))
            
            # 메모리 정리
            del paths, cfr_output, h_freq
            if step % 2000 == 0:
                dr.flush_malloc_cache()
                gc.collect()

        # BS 하나 완료
        bs_np_data = np.array(time_step_data, dtype=np.complex64)
        current_ue_bs_data.append(bs_np_data)
        
        # 맵 청소
        scene.remove("Active_Tx")
        scene.remove("Active_Rx")
        dr.flush_malloc_cache()
        gc.collect()
        print(" 완료.")

    # UE 하나 완료: 데이터 임시 저장
    ue_combined = np.stack(current_ue_bs_data, axis=0) # (3, Time, 64)
    ue_combined = np.transpose(ue_combined, (1, 0, 2)) # (Time, 3, 64)
    
    all_ue_data.append(ue_combined)
    
    # 안전 백업
    np.save(os.path.join(output_dir, f"backup_UE_{ue_idx}.npy"), ue_combined)
    print(f"  💾 UE {ue_idx} 백업 완료.")

# ==========================================
# 6. 최종 병합 및 저장
# ==========================================
print("\n📊 데이터 병합 중...")
final_dataset = np.stack(all_ue_data, axis=0) # (UE, Time, BS, 64)
final_dataset = np.transpose(final_dataset, (1, 0, 2, 3)) # (Time, UE, BS, 64)

print(f"최종 데이터 형태: {final_dataset.shape}")
print(f"최종 데이터 타입: {final_dataset.dtype}")

if np.iscomplexobj(final_dataset):
    print("✅ 최종 확인: 복소수 데이터입니다.")
else:
    print("⚠️ 최종 확인: 실수 데이터입니다.")

np.save(save_path, final_dataset)
print(f"저장 완료: {save_path}")

2026-02-03 17:41:21.183234: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770108081.196248  433125 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770108081.200131  433125 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770108081.210574  433125 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770108081.210584  433125 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770108081.210586  433125 computation_placer.cc:177] computation placer alr